# Train YOLO11n on NEU-DET (steel surface defects) — PS5 Phase 1

**What this does:** downloads the real NEU-DET steel defect images, converts the labels to YOLO format, trains YOLO11n, reports precision / recall / F1 / mAP, and exports `yolo11n_neu.onnx` + `labels.txt` + a test set for the phone.

**How to use:** `Runtime > Change runtime type > T4 GPU`, then run the cells top to bottom (Shift+Enter). Total time is about 30–60 minutes, mostly training.

At the end, download `phone_bundle.zip` (last cell) and send it to the person running the phone.

## 1. Install packages (about 1 min)

In [ ]:
!pip -q install ultralytics onnx onnxslim onnxruntime kagglehub
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| GPU:", torch.cuda.is_available())

## 2. Settings
`IMGSZ = 320` is the recommended input size: NEU images are only 200×200, and 320 runs about 4× faster on the phone than 640. If accuracy is poor, try 640 and compare (write both in the report).

In [ ]:
IMGSZ   = 320      # model input size (320 recommended for the phone; try 640 to compare)
EPOCHS  = 60       # ~20-30 min on a T4 at 320
BATCH   = 32
SEED    = 0
WORK    = "/content/neu"

## 3. Download NEU-DET (real images, ~1,800 photos, 6 classes)
Uses the public Kaggle copy `kaustubhdikshit/neu-surface-defect-database`.
If the download asks for credentials, either upload your `kaggle.json` (Kaggle > Settings > API > Create token) when prompted, or download the zip in a browser and upload it to `/content/` then set `RAW` to the unzipped folder.

In [ ]:
import os, kagglehub
try:
    RAW = kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database")
except Exception as e:
    print("kagglehub failed:", e)
    from google.colab import files
    print("Upload kaggle.json")
    up = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    os.replace("kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
    RAW = kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database")
print("Downloaded to", RAW)
!find "$RAW" -maxdepth 3 -type d | head -20

## 4. Convert Pascal VOC XML labels to YOLO format
Finds every `.xml` and image anywhere in the download, matches them by file name, and keeps the dataset's own train / validation split if it has one (otherwise a fixed 80/20 random split).

In [ ]:
import glob, random, shutil, xml.etree.ElementTree as ET
from pathlib import Path

xmls = glob.glob(f"{RAW}/**/*.xml", recursive=True)
imgs = {Path(p).stem: p for ext in ("jpg","jpeg","png","bmp")
        for p in glob.glob(f"{RAW}/**/*.{ext}", recursive=True)}
print(len(xmls), "xml files,", len(imgs), "images")

def parse(x):
    r = ET.parse(x).getroot()
    W = float(r.find("size/width").text); H = float(r.find("size/height").text)
    objs = []
    for o in r.findall("object"):
        n = o.find("name").text.strip()
        b = o.find("bndbox")
        x1, y1, x2, y2 = (float(b.find(k).text) for k in ("xmin","ymin","xmax","ymax"))
        objs.append((n, x1, y1, x2, y2))
    return W, H, objs

parsed = {}
for x in xmls:
    stem = Path(x).stem
    if stem in imgs:
        parsed[stem] = (x, parse(x))
CLASSES = sorted({o[0] for _, (_, (_, _, objs)) in parsed.items() for o in objs})
print("classes:", CLASSES)
assert len(CLASSES) == 6, "expected 6 NEU classes - check the class names above"

# split: use folder names if the dataset ships train/validation, else random 80/20
def split_of(xml_path):
    p = xml_path.lower()
    if "/valid" in p or "/val/" in p or "/test" in p: return "val"
    if "/train" in p: return "train"
    return None
stems = sorted(parsed)
splits = {s: split_of(parsed[s][0]) for s in stems}
if None in splits.values():
    random.Random(SEED).shuffle(stems)
    cut = int(0.8 * len(stems))
    splits = {s: ("train" if i < cut else "val") for i, s in enumerate(stems)}

if os.path.exists(WORK): shutil.rmtree(WORK)
for sp in ("train","val"):
    os.makedirs(f"{WORK}/images/{sp}"); os.makedirs(f"{WORK}/labels/{sp}")
for s in stems:
    sp = splits[s]; xml_path, (W, H, objs) = parsed[s]
    src = imgs[s]; dst = f"{WORK}/images/{sp}/{s}{Path(src).suffix}"
    shutil.copy(src, dst)
    with open(f"{WORK}/labels/{sp}/{s}.txt", "w") as f:
        for n, x1, y1, x2, y2 in objs:
            cx, cy = (x1+x2)/2/W, (y1+y2)/2/H; w, h = (x2-x1)/W, (y2-y1)/H
            f.write(f"{CLASSES.index(n)} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")

with open(f"{WORK}/data.yaml","w") as f:
    f.write(f"path: {WORK}\ntrain: images/train\nval: images/val\nnames:\n")
    for i, c in enumerate(CLASSES): f.write(f"  {i}: {c}\n")
print({sp: len(os.listdir(f'{WORK}/images/{sp}')) for sp in ('train','val')})

## 5. Look at a few labelled images (sanity check)

In [ ]:
import cv2, matplotlib.pyplot as plt
sample = sorted(os.listdir(f"{WORK}/images/train"))[::300][:6]
plt.figure(figsize=(15,3))
for i, fn in enumerate(sample):
    im = cv2.imread(f"{WORK}/images/train/{fn}"); H, W = im.shape[:2]
    for ln in open(f"{WORK}/labels/train/{Path(fn).stem}.txt"):
        c, cx, cy, w, h = ln.split(); cx, cy, w, h = map(float, (cx, cy, w, h))
        cv2.rectangle(im, (int((cx-w/2)*W), int((cy-h/2)*H)), (int((cx+w/2)*W), int((cy+h/2)*H)), (0,0,255), 1)
    plt.subplot(1,6,i+1); plt.imshow(im[:,:,::-1]); plt.title(CLASSES[int(c)], fontsize=8); plt.axis("off")
plt.show()

## 6. Train YOLO11n (about 20–30 min on a T4)

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")        # COCO-pretrained nano model, fine-tuned on NEU
results = model.train(data=f"{WORK}/data.yaml", imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH,
                      seed=SEED, patience=20, project="/content/runs", name="neu_yolo11n",
                      exist_ok=True, fliplr=0.5, flipud=0.5, degrees=0, mosaic=1.0)
BEST = "/content/runs/neu_yolo11n/weights/best.pt"

## 7. Metrics for the report: precision, recall, F1, mAP (overall and per class)

In [ ]:
import pandas as pd
best = YOLO(BEST)
m = best.val(data=f"{WORK}/data.yaml", imgsz=IMGSZ, split="val", plots=True)
P, R = float(m.box.mp), float(m.box.mr)
F1 = 2*P*R/(P+R) if P+R else 0
print(f"Overall  P={P:.3f}  R={R:.3f}  F1={F1:.3f}  mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}")
rows = []
for i, c in enumerate(CLASSES):
    p, r, ap50, ap = m.box.class_result(i)
    rows.append({"class": c, "precision": round(p,3), "recall": round(r,3),
                 "F1": round(2*p*r/(p+r),3) if p+r else 0, "mAP50": round(ap50,3), "mAP50-95": round(ap,3)})
df = pd.DataFrame(rows); display(df)
df.to_csv("/content/metrics_per_class.csv", index=False)
print("Confusion matrix and PR curves are in", m.save_dir)

## 8. Export to ONNX for the phone
Fixed input size, opset 12, no NMS inside the model (the phone code does NMS itself). Output shape should be `(1, 10, N)` = 4 box values + 6 class scores.

In [ ]:
onnx_path = best.export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True, dynamic=False)
shutil.copy(onnx_path, "/content/yolo11n_neu.onnx")
with open("/content/labels.txt","w") as f: f.write("\n".join(CLASSES) + "\n")
import onnxruntime as ort, numpy as np
s = ort.InferenceSession("/content/yolo11n_neu.onnx", providers=["CPUExecutionProvider"])
o = s.run(None, {s.get_inputs()[0].name: np.zeros((1,3,IMGSZ,IMGSZ), np.float32)})[0]
print("ONNX output shape:", o.shape, "| size MB:", round(os.path.getsize('/content/yolo11n_neu.onnx')/1e6, 1))

## 9. Build the phone bundle
`phone_bundle.zip` contains the model, labels, the validation images as `test_images/` and their ground truth as `test_labels/` (for the benchmark's precision / recall on the phone), plus the metrics files for the report.

In [ ]:
B = "/content/phone_bundle"
if os.path.exists(B): shutil.rmtree(B)
shutil.copytree(f"{WORK}/images/val", f"{B}/test_images")
shutil.copytree(f"{WORK}/labels/val", f"{B}/test_labels")
for fn in ("yolo11n_neu.onnx", "labels.txt", "metrics_per_class.csv"): shutil.copy(f"/content/{fn}", B)
os.makedirs(f"{B}/report_figures", exist_ok=True)
for fn in glob.glob(f"{m.save_dir}/*.png") + glob.glob("/content/runs/neu_yolo11n/results.png"):
    shutil.copy(fn, f"{B}/report_figures/")
shutil.make_archive("/content/phone_bundle", "zip", B)
print("bundle MB:", round(os.path.getsize('/content/phone_bundle.zip')/1e6, 1))
from google.colab import files
files.download("/content/phone_bundle.zip")